# Chapter 58: Error Analysis, Interpretation, Responsibility, and Deployment

Turn NRG package-risk predictions into an error inventory and release gate.


In [ ]:
from pathlib import Path
import sys,numpy as np,matplotlib.pyplot as plt
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.responsibility import *
print('Imports ready.')


Imports ready.


In [ ]:
actual=np.array([0,1,0,1,1,0,1,0,1,0,1,0]);prob=np.array([.1,.8,.7,.2,.9,.4,.55,.6,.3,.15,.75,.45]);factory=np.array(['A']*6+['B']*6);labels=error_labels(actual,prob)
print('Outcomes:',{k:int(np.sum(labels==k)) for k in ['TP','TN','FP','FN']})


Outcomes: {'TP': 4, 'TN': 4, 'FP': 2, 'FN': 2}


In [ ]:
rows=slice_error_rates(actual,prob,factory)
for row in rows:print(f"Factory {row['group']}: n={row['count']}, error={row['error_rate']:.1%}, FNR={row['false_negative_rate']:.1%}")


Factory A: n=6, error=33.3%, FNR=16.7%
Factory B: n=6, error=33.3%, FNR=16.7%


In [ ]:
queue=review_queue(prob,4);print('Review indices:',queue.tolist());checks={'owner':'Quality','intended_use':'inspection priority','test_slices':rows,'fallback':'manual','monitoring':'weekly','rollback':''};report=readiness_report(checks);print('Ready:',report['ready']);print('Missing:',report['missing'])


Review indices: [6, 11, 5, 7]
Ready: False
Missing: ['rollback']


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4));order=['TP','TN','FP','FN'];axes[0].bar(order,[np.sum(labels==k) for k in order],color=['#287d4f','#6082a5','#d8943c','#b84b4b']);axes[0].set(title='Error inventory',ylabel='Packages');axes[1].scatter(np.arange(len(prob)),prob,c=np.isin(labels,['FP','FN']),cmap='coolwarm',s=65);axes[1].axhline(.5,color='black',ls='--');axes[1].scatter(queue,prob[queue],facecolors='none',edgecolors='gold',s=150,label='review');axes[1].set(title='Probabilities and review queue',xlabel='Package',ylabel='Risk probability');axes[1].legend();fig.tight_layout();plt.show()


## Interpretation

The same aggregate error rate can hide different mechanisms, so every slice needs outcome counts and examples. Deployment remains blocked until the rollback control is defined and tested.


In [ ]:
# Practice: change the review capacity and compare selected packages.
